# [UAV] BỘ 3: Dataset_Bo3_5Class_IITU_Drone_Benchmark
### Đồ án Tốt nghiệp Capstone AI - Báo cáo Tiến độ Task 2 (Drone Multi-Angle & Lighting)
- **Tác giả / Chủ trì đồ án**: Nguyễn Hàn Như (SE183674)
- **GVHD**: ThS. Nguyễn Xuân Huy | **Đơn vị**: Đại học FPT
- **Kiến trúc đánh giá**: Rep-YOLO11s vs Baseline YOLO11s
- **Tập dữ liệu**: IITU Safety-Helmet (5 Colors: `blue`, `green`, `red`, `white`, `yellow`)
- **Đường dẫn Kaggle**: `/kaggle/input/datasets/dngtrnphmtunse183674/iitu-safety-helmet` (hoặc `/kaggle/input/iitu-safety-helmet`)

---

## [OBJECTIVE] GIẢI QUYẾT CÁC YÊU CẦU CỐT LÕI CỦA TASK 2 (REVIEW 2):
1. **Giai đoạn 1 (Dataset Pipeline)**:
   - Thống kê nguồn dữ liệu: 1,669 ảnh flycam/drone và mặt đất từ ĐH IITU Kazakhstan.
   - Góc quay đa dạng: 0° (ngang), 45° (nghiêng), 90° (thẳng đứng từ trên nóc).
   - Chiếu sáng: Ban ngày (daytime) và ban đêm (night lamp light).
   - Thống kê phân bố lớp: `white` (1,401 - 21.6%), `yellow` (1,362 - 21.0%), `blue` (1,359 - 21.0%), `red` (1,216 - 18.8%), `green` (1,133 - 17.5%).
   - Tỷ lệ mất cân bằng cực thấp (1.24:1).
2. **Giai đoạn 2 (Baseline & Proposed Execution)**:
   - Kiểm tra tính ổn định nhận diện màu sắc mũ từ góc quay flycam trên cao.


In [ ]:
# CELL 1: KIỂM TRA PHẦN CỨNG & CẤU HÌNH GPU TESLA T4
import os
import sys
import torch

print("=" * 80)
print("[EXECUTION] HỆ THỐNG THỰC NGHIỆM CAPSTONE AI - BỘ 3 (IITU DRONE 5-CLASS)")
print("=" * 80)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"Số lượng GPU khả dụng: {device_count}")
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")


In [ ]:
# CELL 2: PHÁT HIỆN TẬP DỮ LIỆU IITU SAFETY-HELMET
import glob
from pathlib import Path
from collections import Counter
import random
import shutil

print("=" * 80)
print("[INSPECTION] TÌM KIẾM TẬP DỮ LIỆU IITU SAFETY-HELMET TRONG /kaggle/input/...")
print("=" * 80)

search_paths = [
    "/kaggle/input/datasets/dngtrnphmtunse183674/iitu-safety-helmet",
    "/kaggle/input/iitu-safety-helmet",
    "/kaggle/input/dngtrnphmtunse183674/iitu-safety-helmet"
]

iitu_root = None
for p in search_paths:
    if os.path.exists(p):
        iitu_root = p
        break

if not iitu_root:
    for root, dirs, files in os.walk("/kaggle/input"):
        if any("iitu" in d.lower() for d in dirs):
            iitu_root = os.path.join(root, [d for d in dirs if "iitu" in d.lower()][0])
            break

print(f"[PATH] Thư mục IITU tìm thấy: {iitu_root}")

# Quét ảnh và annotation
all_images = glob.glob(os.path.join(iitu_root, "**", "*.jpg"), recursive=True) +              glob.glob(os.path.join(iitu_root, "**", "*.png"), recursive=True)
all_txts = glob.glob(os.path.join(iitu_root, "**", "*.txt"), recursive=True)

valid_txts = [t for t in all_txts if not any(k in os.path.basename(t).lower() for k in ['classes', 'readme', 'notes'])]
print(f"Tổng số ảnh: {len(all_images)} | Tổng số txt nhãn: {len(valid_txts)}")


In [ ]:
# CELL 3: CHUẨN HÓA SANG CẤU TRÚC YOLO TRAIN/VAL/TEST (80/10/10)
TARGET_DIR = Path("/kaggle/working/IITU_5CLASS_STANDARDIZED")
for split in ['train', 'val', 'test']:
    (TARGET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (TARGET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Map txt by stem
txt_map = {Path(t).stem: t for t in valid_txts}
valid_samples = []
class_counter = Counter()

for img_p in all_images:
    stem = Path(img_p).stem
    if stem not in txt_map:
        continue
    
    txt_p = txt_map[stem]
    with open(txt_p, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if len(l.strip().split()) >= 5]
    
    if lines:
        for l in lines:
            c_id = l.split()[0]
            class_counter[c_id] += 1
        valid_samples.append((img_p, lines))

print(f"Tổng số ảnh có nhãn hợp lệ: {len(valid_samples)}")
print("Phân bố bounding box (IITU Raw Class IDs 0-4):", dict(class_counter))

# Phân chia 80/10/10
random.seed(42)
random.shuffle(valid_samples)
n = len(valid_samples)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

splits = {
    'train': valid_samples[:n_train],
    'val': valid_samples[n_train:n_train+n_val],
    'test': valid_samples[n_train+n_val:]
}

for split_name, samples in splits.items():
    print(f"Đang ghi split '{split_name}': {len(samples)} mẫu...")
    for img_p, lines in samples:
        dst_img = TARGET_DIR / 'images' / split_name / Path(img_p).name
        dst_lbl = TARGET_DIR / 'labels' / split_name / f"{Path(img_p).stem}.txt"
        shutil.copy2(img_p, dst_img)
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines) + '\n')

yaml_content = f"""# IITU Safety-Helmet 5-Class Dataset Configuration
path: {TARGET_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 5
names: ['blue_helmet', 'green_helmet', 'red_helmet', 'white_helmet', 'yellow_helmet']
"""

with open(TARGET_DIR / 'data.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"[VERIFIED] Chuẩn hóa hoàn tất! data.yaml đã sẵn sàng tại: {TARGET_DIR / 'data.yaml'}")


In [ ]:
# CELL 4: HUẤN LUYỆN THỬ NGHIỆM ĐÁNH GIÁ (5 EPOCHS VERIFICATION)
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

from ultralytics import YOLO
import time

print("=" * 80)
print("[EXECUTION] KHỞI CHẠY HUẤN LUYỆN BENCHMARK IITU 5-CLASS TRÊN GPU T4")
print("=" * 80)

model = YOLO("yolo11s.pt")

start_time = time.time()
results = model.train(
    data=str(TARGET_DIR / 'data.yaml'),
    epochs=5,
    imgsz=640,
    batch=16,
    seed=42,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs",
    name="iitu_5class_run",
    exist_ok=True,
    verbose=True
)
train_duration = time.time() - start_time
print(f"[TIME] Thời gian huấn luyện 5 epochs: {train_duration:.2f}s ({train_duration/60:.2f} phút)")


In [ ]:
# CELL 5: ĐÁNH GIÁ ĐỘC LẬP TẬP TEST IITU (GÓC DRONE 0°, 45°, 90°)
print("=" * 80)
print("[EVALUATION] ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP IITU")
print("=" * 80)

metrics = model.val(
    data=str(TARGET_DIR / 'data.yaml'),
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu"
)

class_names = ['blue_helmet', 'green_helmet', 'red_helmet', 'white_helmet', 'yellow_helmet']
p = metrics.box.mp
r = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print(f"\n[OBJECTIVE] KẾT QUẢ TỔNG QUAN:")
print(f"  Precision: {p*100:.2f}%")
print(f"  Recall:    {r*100:.2f}%")
print(f"  mAP@50:    {map50*100:.2f}%")
print(f"  mAP@50-95: {map50_95*100:.2f}%")

print(f"\n[REPORT] KẾT QUẢ CHI TIẾT THEO TỪNG MÀU MŨ:")
for i, name in enumerate(class_names):
    c_p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    c_r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    c_map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    print(f"  [{i}] {name:<15}: P={c_p*100:.2f}% | R={c_r*100:.2f}% | mAP50={c_map50*100:.2f}%")


In [ ]:
# CELL 6: TẠO FILE BÁO CÁO KẾT QUẢ IITU
report_md = f'''# [REPORT] BÁO CÁO KẾT QUẢ THỰC NGHIỆM TASK 2 - BỘ 3 (IITU DRONE 5-CLASS)
**Đồ án Tốt nghiệp Capstone AI** - SV: Nguyễn Hàn Như | GVHD: ThS. Nguyễn Xuân Huy

### 1. Thông số Dataset IITU sau chuẩn hóa:
- **Tổng số ảnh hợp lệ**: {len(valid_samples)} ảnh
- **Tập Train**: {len(splits['train'])} ảnh (80%)
- **Tập Val**:   {len(splits['val'])} ảnh (10%)
- **Tập Test**:  {len(splits['test'])} ảnh (10%)

### 2. Kết quả Đánh giá Benchmark trên Tập Test Độc lập:
| Chỉ số (Metric) | Giá trị thực nghiệm |
| :--- | :---: |
| **Precision** | **{p*100:.2f}%** |
| **Recall** | **{r*100:.2f}%** |
| **mAP@50** | **{map50*100:.2f}%** |
| **mAP@50-95** | **{map50_95*100:.2f}%** |

### 3. Đánh giá chi tiết 5 màu mũ bảo hộ:
| Class ID | Màu Mũ Bảo Hộ | Precision (%) | Recall (%) | mAP@50 (%) |
| :---: | :--- | :---: | :---: | :---: |
| 0 | `blue_helmet` (Xanh dương) | {metrics.box.p[0]*100:.2f} | {metrics.box.r[0]*100:.2f} | {metrics.box.ap50[0]*100:.2f} |
| 1 | `green_helmet` (Xanh lá) | {metrics.box.p[1]*100:.2f} | {metrics.box.r[1]*100:.2f} | {metrics.box.ap50[1]*100:.2f} |
| 2 | `red_helmet` (Đỏ) | {metrics.box.p[2]*100:.2f} | {metrics.box.r[2]*100:.2f} | {metrics.box.ap50[2]*100:.2f} |
| 3 | `white_helmet` (Trắng) | {metrics.box.p[3]*100:.2f} | {metrics.box.r[3]*100:.2f} | {metrics.box.ap50[3]*100:.2f} |
| 4 | `yellow_helmet` (Vàng) | {metrics.box.p[4]*100:.2f} | {metrics.box.r[4]*100:.2f} | {metrics.box.ap50[4]*100:.2f} |

### 4. Kết luận khoa học:
1. IITU là tập dữ liệu tuyệt vời để nhận diện màu sắc mũ từ góc quay flycam/drone trên cao (0°, 45°, 90°) và môi trường ban đêm.
2. Tuy nhiên, tập này HOÀN TOÀN KHÔNG CÓ nhãn người không đội mũ (`none`) và áo bảo hộ (`vest`).
3. Khuyến nghị: Phải dùng phương án Bộ 4 (Tổng hợp GDUT + IITU + Vest) để giải quyết toàn diện bài toán bảo vệ an toàn công trường.
"""

with open("/kaggle/working/BAO_CAO_BO3_IITU_5CLASS.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print("[VERIFIED] Đã xuất báo cáo: /kaggle/working/BAO_CAO_BO3_IITU_5CLASS.md")
